In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual, Layout
import ipywidgets as widgets

In [ ]:
import io
import pandas as pd
import xgboost as xgb 
import pickle
import numpy as np
import os
from tqdm import tqdm

In [ ]:
np.set_printoptions(precision=4)
from warnings import filterwarnings
filterwarnings("ignore")

In [ ]:
from IPython.display import Markdown, display, clear_output
from ipywidgets import Layout, Button, Box, VBox

def printmd(string, color=None,size='20'):
    colorstr = "<span style='color:{};font-size:{}px'>{}</span>".format(color, size, string)
    display(Markdown(colorstr))
import matplotlib.pyplot as plt

In [ ]:
np.set_printoptions(precision=10)

In [ ]:
import pickle

In [ ]:
with open('./model_a1.pkl','rb') as f:
    load_dict = pickle.load(f)
f.close()

with open('./model_a2.pkl','rb') as f:
    load_dict2 = pickle.load(f)
f.close()

model = load_dict['model']
model2 = load_dict2['model']

In [ ]:
coslist = ['$f_{cu}$', '$\epsilon_{cu}$', '$f_y$', '$E$', '$b$', '$cR_1$', '$PGA$', '$PGV$', '$Vs_{30}$', '$Tp\ (s)$', '$M_W$', '$R_{hyp}$', '$R$', '$NOSt$', '$t_{1}$', '$z/h$']


In [ ]:
from format_font import *
set_font(size=20)

In [ ]:
bad = 0

In [ ]:
rdict = {'Elastic':0.1, '1':1,'2':2,'3':3,'4':4,'5':5}

In [ ]:
def colseabornext(text='text', color='red'):
    printmd(f"<font color='{color}'>{text}</font>")
@interact
def predict(fcu = widgets.Combobox(value = '37.06', description = '<b style="font-size:17px;">$f_{cu}$</b>', disabled = False,placeholder='24 - 69 MPa',layout=Layout(width='25%')),
         ecu = widgets.Combobox(value = '0.0029', description = '<b style="font-size:17px;">$\epsilon_{cu}$</b>', disabled = False, placeholder='0.0010 - 0.0035',layout=Layout(width='25%')),
         fy = widgets.Combobox(value = '520.9', description = '<b style="font-size:17px;">$f_y$</b>', disabled = False, placeholder='462 - 585 MPa',layout=Layout(width='25%')),
         E = widgets.Combobox(value = '218.5', description = '<b style="font-size:17px;">$E$</b>', disabled = False, placeholder='201 - 223 GPa',layout=Layout(width='25%')),
         b = widgets.Combobox(value = '0.019', description = '<b style="font-size:17px;">$b$</b>', disabled = False, placeholder='0.014 - 0.025',layout=Layout(width='25%')),
         cr1 = widgets.Combobox(value = '0.894', description = '<b style="font-size:17px;">$cR_1$</b>', disabled = False, placeholder='0.87 - 0.91',layout=Layout(width='25%')),
            
         PGA = widgets.Combobox(value = '0.704', description = '<b style="font-size:17px;">$PGA$</b>', disabled = False,placeholder='PGA',layout=Layout(width='25%')),
         PGV = widgets.Combobox(value = '68.3', description = '<b style="font-size:17px;">$PGV$</b>', disabled = False, placeholder='PGV',layout=Layout(width='25%')),
         Vs30 = widgets.Combobox(value = '489.34', description = '<b style="font-size:17px;">$Vs_{30}$</b>', disabled = False, placeholder='Vs30',layout=Layout(width='25%')),
         Tp = widgets.Combobox(value = '0.81', description = '<b style="font-size:17px;">$T_p$</b>', disabled = False, placeholder='Pulse period',layout=Layout(width='25%')),
         Mw = widgets.Combobox(value = '5.8', description = '<b style="font-size:17px;">$M_w$</b>', disabled = False, placeholder='Magnitude',layout=Layout(width='25%')),
         Rhyp = widgets.Combobox(value = '13.5', description = '<b style="font-size:17px;">$R_{hyp}$</b>', disabled = False, placeholder='Hypocentral distance (km)',layout=Layout(width='25%')),
            
         NOSt = widgets.Combobox(value = '4', description = '<b style="font-size:17px;">$NOSt$</b>', disabled = False, placeholder='4, 8, 12',layout=Layout(width='25%')),
         t1 = widgets.Combobox(value = '0.66', description = '<b style="font-size:17px;">$T_{1}$</b>', disabled = False, placeholder='Fundamental period',layout=Layout(width='25%')),
            
         
        # Pulse = widgets.RadioButtons(options=['Pulse', 'Non Pulse'],description = '<b style="font-size:17px;">GM Type</b>'),
        Category = widgets.Combobox(value = 'Elastic', description = '<b style="font-size:17px;">$R$</b>', disabled = False, placeholder='Elastic, 1, 2, 3, 4, 5',layout=Layout(width='25%'))):#,

    R = Category

    Model = widgets.ToggleButton(
    options=['Numerical value', 'Graphical view'],
    description = '<b style="font-size:17px;">Model</b>',
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltips=['Numerical value', 'Graphical view'],
                    style=dict(font_weight='bold'))

        
    
    matvec = [fcu, ecu, fy, E, b, cr1]
    matvec = [float(m) for m in matvec]
    
    gmvec = [PGA, PGV, Vs30, Tp, Mw, Rhyp]
    gmvec = [float(g) for g in gmvec]
    
    rdict = {'Elastic':0.1, '1':1,'2':2,'3':3,'4':4,'5':5}
    strvec = [rdict[R], int(NOSt), float(t1)]
    
    NOSt = strvec[-2]
    zbyh = np.arange(1,NOSt+1)/NOSt
    
    zhvec = list(zbyh)
    invec = NOSt*[matvec + gmvec + strvec]
    
    dfinput = pd.DataFrame(invec, columns=coslist[:-1])
    dfinput[coslist[-1]] = zhvec

    model = load_dict['model']
    model2 = load_dict2['model']
    
    xin = dfinput.values
    dfinput['prediction'] = model.predict(xin)
    dfinput['prediction2'] = model2.predict(xin)

    b1 = widgets.Button(description='Get A1 and A2',tooltip = 'Description', icon = 'fa-table',style=dict(font_weight='bold', button_color='lightseagreen'),
                               layout=Layout(width='25%', height='40px'))
    b1_output = widgets.Output()

    def b1_clicked(b):
        with b1_output:
            clear_output()
            # print('numercial values')
            try:
                
                dshow = dfinput
                print('\n\n\n')
                bad = dshow.copy()
                dshow.index  = np.arange(1,int(NOSt+1))
                dshow['z/h'] = dshow['$z/h$']
                dshow['A1'] = dshow.prediction.round(3)
                dshow['A2'] = dshow.prediction2.round(3)
                
                display(dshow[['z/h','A1', 'A2']])
            except Exception as e:
                print(e)
                
   
    b1.on_click(b1_clicked)
    
    results = widgets.HBox([b1, b1_output])
    
    display(results)